# `load_documents_from_gcs` — pipeline-এর প্রথম কাজ

আচ্ছা, এখন আসল জায়গায় এসেছি। আমি পাশে বসে একটা একটা করে বুঝিয়ে দিচ্ছি — তাড়াহুড়ো নেই।

এর আগে যা ছিল (config, prompts, logging) সব প্রস্তুতি। এখান থেকে আসল data ঢোকা শুরু: cloud-এ পড়ে থাকা text file।

এক লাইনে মনে রাখো, বাকিটা এই নোটের ডিটেইল:

**এই function cloud-এর text file নেয়, আর LangChain যেভাবে বোঝে সেই `Document` object বানিয়ে দেয়।**

নিচের cell গুলো পড়ার সাথে সাথে চালিয়ে দেখো। যেগুলো cloud ছাড়াই বোঝা যায়, সেগুলো এখনি চলবে। একদম শেষেরটাই আসল GCS call।

---

## অংশ ১ — Signature আর default argument

Function-এর মাথায় দুইটা parameter। দুইটারই default আছে, তাই খালি ডাকলেও চলে।

```python
def load_documents_from_gcs(
    bucket_name: str = config.GCS_BUCKET_NAME,
    prefix: str = config.GCS_PREFIX,
) -> list[Document]:
```

তিনভাবে ডাকতে পারো। নিচের cell-এ একই তিনটা ডাক চালিয়ে দেখো — return হচ্ছে কোন bucket আর কোন prefix সে ধরল।

```python
load_documents_from_gcs()                              # config থেকে দুইটাই
load_documents_from_gcs(prefix="hr_policies/test/")    # শুধু prefix বদলালাম
load_documents_from_gcs("other-bucket", "raw/")        # দুইটাই নিজে দিলাম
```

এই design ইচ্ছে করে এমন। রোজকার কাজে কিছু না ভেবে `load_documents_from_gcs()` লিখলেই হয়। Notebook-এ ছোট একটা prefix দিয়ে ৩টা file নিয়ে খেলতে চাইলে শুধু ওই argument পাঠাও। Production-এর code-এ একটা অক্ষরও বদলাতে হয় না।

`-> list[Document]` বলছে ফেরত আসবে Document-এর list। Python 3.9+ এ `list[Document]` সরাসরি লেখা যায়। তার আগের version-এ `typing` থেকে `List` import করতে হতো।

In [ ]:
# শুধু তিন রকম ডাক দেখানো। আসল function শেষের cell-এ import হয়।
def load_documents_from_gcs(
    bucket_name: str = "hr-policies-bucket",   # config.GCS_BUCKET_NAME-এর জায়গায়
    prefix: str = "hr_policies/raw/",          # config.GCS_PREFIX-এর জায়গায়
):
    return bucket_name, prefix

print(load_documents_from_gcs())                            # config থেকে দুইটাই
print(load_documents_from_gcs(prefix="hr_policies/test/"))  # শুধু prefix বদলালাম
print(load_documents_from_gcs("other-bucket", "raw/"))      # দুইটাই নিজে দিলাম

### এখানে একটা ফাঁদ আছে, আমিও একবার পড়েছিলাম

Default value প্রতিবার call-এর সময় নতুন করে পড়া হয় না। Function **define হওয়ার মুহূর্তে একবার** hisab হয়, তারপর সেই মান function-এর গায়ে আটকে থাকে।

`document_loader.py` প্রথমবার import হলে Python তখনই `config.GCS_BUCKET_NAME` দেখে নেয়। পরে notebook-এ environment বদলালে function সেটা আর দেখে না।

```python
import os
os.environ["GCS_BUCKET_NAME"] = "new-bucket"   # import করার পরে বদলালাম
load_documents_from_gcs()                       # এখনও পুরনো bucket-এই যাবে
```

নিচের cell-টা চালাও। প্রথম print দেখাবে মানটা definition-এর সময়ই সিল হয়ে গেছে। Environment `new-bucket` করলেও খালি ডাক এখনও `old-bucket` দেয়। নতুন bucket চাইলে সরাসরি argument দাও: `load_documents_from_gcs(bucket_name="new-bucket")`।

Notebook-এ config বদলে কিছু হচ্ছে না মনে হলে kernel restart করো, অথবা argument পাঠাও। "Code বদলালাম, কাজ করছে না কেন" — এই ঘণ্টাখানেকটা প্রায়ই এখান থেকেই আসে। এটা Python-এর নিয়ম, তোমার ভুল না।

In [ ]:
import os

previous = os.environ.get("GCS_BUCKET_NAME")
os.environ["GCS_BUCKET_NAME"] = "old-bucket"

def load_documents_from_gcs(bucket_name: str = os.environ["GCS_BUCKET_NAME"]):
    return bucket_name

print("sealed at definition:", load_documents_from_gcs.__defaults__)

os.environ["GCS_BUCKET_NAME"] = "new-bucket"          # define হওয়ার পরে বদলালাম
print(load_documents_from_gcs())                      # এখনও old-bucket
print(load_documents_from_gcs(bucket_name="new-bucket"))

if previous is None:
    os.environ.pop("GCS_BUCKET_NAME", None)
else:
    os.environ["GCS_BUCKET_NAME"] = previous

---

## অংশ ২ — Client তৈরি

এবার Google-এর সাথে কথা বলার লাইন।

```python
client = storage.Client(project=config.PROJECT_ID)
```

`Client` হলো GCS-এর সাথে তোমার মুখপাত্র। এর কাছে তিনটা জিনিস থাকে: তুমি কে (credential), কোন project-এর বিল কাটা হবে, আর একটা HTTP session যাতে বারবার নতুন করে সংযোগ না খুলতে হয়।

মজার ব্যাপার, তুমি এই file-এ password লেখোনি। তাও login হয়। Library নিজে **ADC (Application Default Credentials)** ধরে খোঁজে, এই ক্রমে:

1. `GOOGLE_APPLICATION_CREDENTIALS`-এ service-account JSON-এর path আছে কি না
2. এই মেশিনে `gcloud auth application-default login` চালানো আছে কি না
3. Code যদি Cloud Run, GCE, বা Vertex AI-এর ভেতরে চলে, সেখানকার attached service account

`DefaultCredentialsError` দেখলে বুঝবে এই তিনটার একটাও মেলেনি। Terminal-এ `gcloud auth application-default login` চালালে সাধারণত মিটে যায়।

`project=` দরকার, কারণ একই login দিয়ে একাধিক project-এ ঢোকা যায়। এই argument বলে দেয় quota আর billing কোন project-এর।

তারপর:

```python
bucket = client.bucket(bucket_name)
```

এই line-টা নিয়ে অনেকে ভয় পায়, কিন্তু **এটা network-এ যায় না**। শুধু একটা local object বানায়, যার ভেতরে লেখা "আমি এই নামের bucket-কে বোঝাচ্ছি"। Bucket আছে কি না, তোমার access আছে কি না — এখনও কিছু চেক হয়নি।

| Code | Network call? | কখন error দেবে |
|---|---|---|
| `client.bucket("nonexistent")` | না | কখনো না — চুপচাপ object বানাবে |
| `client.get_bucket("nonexistent")` | হ্যাঁ | সাথে সাথে `NotFound` |

তাই নাম ভুল হলে এই line চুপ করে থাকবে। Error আসবে পরে, `list_blobs()` যখন সত্যি server-এ যাবে। Traceback-এর line number দেখে "এই line-ই দোষী" ভেবো না — দোষটা একটু আগের নামটায়।

---

## অংশ ৩ — Accumulator pattern

```python
documents = []
```

খালি একটা list। Loop-এর ভেতরে একটা একটা করে জমবে, শেষে পুরো list return হবে।

এই ধরনের নাম **accumulator**: খালি পাত্র, তারপর loop, তারপর return। Pipeline-এ এই ছবিটা বারবার আসবে, তাই এখন চোখ সয়ে যাক। নিচের cell-এ list কীভাবে বাড়ছে সেটা print করা আছে।

In [ ]:
documents = []
for filename in ("wfh.txt", "leave.txt", "travel.txt"):
    documents.append(filename)
    print(len(documents), documents)

---

## অংশ ৪ — Blob-এর উপর loop

```python
for blob in bucket.list_blobs(prefix=prefix):
```

এই এক লাইনে তিনটা আলাদা ধারণা লুকিয়ে আছে। একটা একটা করে নাও।

### Blob কী

GCS "file" বলে না, বলে **blob** (বা object)। একটা blob-এ থাকে তার নাম, তার বাইট, আর একটু metadata — size, content-type, কখন তৈরি হয়েছে।

### GCS-এ folder নেই, দেখতে folder-এর মতো

Console-এ folder দেখলেও ভেতরে namespace একদম **flat**। Blob-এর নামটাই পুরো path:

```
hr_policies/raw/work_from_home_policy.txt
```

এই `/` কোনো বিশেষ চিহ্ন না। নামের একটা অক্ষর, ব্যস। Console `/` দেখে folder এঁকে দেখায়, যাতে আমাদের চোখ শান্ত থাকে।

### Prefix মানে "এই অক্ষর দিয়ে যা শুরু"

`prefix="hr_policies/raw/"` মানে folder-এ ঢোকা না। মানে **যে নাম এই অক্ষরগুলো দিয়ে শুরু, সেগুলো দাও**। তাই `hr_policies/raw/2024/old.txt`-ও আসবে — সেও ওই দিয়ে শুরু।

শেষে `/` রাখাটা ছোট জিনিস, কিন্তু বাঁচায়। `prefix="hr"` দিলে `hr_policies/...` এর সাথে `hr_backup_old/...`-ও চলে আসত। নিচের cell-এ দুটো prefix পাশাপাশি চালাও, পার্থক্যটা চোখে পড়বে।

In [ ]:
names = [
    "hr_policies/raw/work_from_home_policy.txt",
    "hr_policies/raw/2024/old.txt",
    "hr_backup_old/secret.txt",
    "hr_policies/raw/",
]

for prefix in ("hr", "hr_policies/raw/"):
    print(repr(prefix))
    for name in names:
        if name.startswith(prefix):
            print(" ", name)
    print()

### `list_blobs()` একসাথে সব আনে না

এটা পুরো list memory-তে তুলে আনে না। **Lazy iterator** — loop যতবার ঘোরে, ততবার পরের item দেয়। ভেতরে page করে API call হয়, default-এ ১০০০টা করে।

Notebook-এ এখানে একবার হোঁচট খাবে, সবাই খায়:

```python
blobs = bucket.list_blobs(prefix="hr_policies/raw/")
print(len(blobs))          # TypeError — iterator-এর len নেই
print(list(blobs))         # কাজ করে
print(list(blobs))         # দ্বিতীয়বার খালি
```

দ্বিতীয়বার খালি, কারণ iterator একবার শেষ হয়ে গেলে আর কিছু অবশিষ্ট থাকে না। বারবার দরকার হলে একবারে list বানিয়ে রাখো: `blobs = list(bucket.list_blobs(...))`।

নিচের cell আসল cloud না, একই অভ্যাসের একটা ছোট iterator। `len` ধরতে গেলে TypeError, একবার `list` করলে নামগুলো, আবার করলে `[]`।

In [ ]:
# bucket.list_blobs() একই রকম iterator — এখানে ছোট একটা yield দিয়ে দেখানো
def list_blobs():
    for name in (
        "hr_policies/raw/work_from_home_policy.txt",
        "hr_policies/raw/leave_policy.txt",
    ):
        yield name

blobs = list_blobs()
try:
    print(len(blobs))
except TypeError as error:
    print(type(error).__name__ + ":", error)

blobs = list_blobs()
print(list(blobs))
print(list(blobs))   # একই iterator আবার — খালি

---

## অংশ ৫ — ফিল্টার

```python
if not blob.name.endswith(".txt"):
    continue
```

`continue` বলে "এই blob-টা থাক, পরেরটাতে যাও"। `break` হলে পুরো loop থেমে যেত। এখানে আমরা বাকি file-গুলোও দেখতে চাই, তাই `continue`।

`not` সহ লাইনটা উল্টো করে পড়ো: **`.txt` দিয়ে শেষ না হলে বাদ।**

`list_blobs` prefix-এর নিচে যা পায় সব দেয়। `.pdf`, `.docx`, এমনকি Console থেকে folder বানালে যে ফাঁকা placeholder তৈরি হয় (`hr_policies/raw/`, নাম slash দিয়ে শেষ) — সেটাও। ওগুলো `.txt` না, তাই এই লাইনেই ছেঁকে যায়। Download-এর আগেই, তাই অযথা টাকা বা সময় যায় না।

PDF আর DOCX এখানে ইচ্ছে করে বাদ। সেগুলো `processor.py` (05) পড়ে। এই file-এর docstring-এর মানেও সেটাই: **যে text আগে থেকেই text, শুধু সেটাই এখানে লোড হয়।**

একটা চুপচাপ ফাঁক: `.endswith(".txt")` **case-sensitive**। কেউ `POLICY.TXT` upload করলে কোনো error ছাড়াই বাদ পড়ে যাবে। চাইলে `blob.name.lower().endswith(".txt")`। নিচের cell-এ দুটোই পাশাপাশি।

In [ ]:
names = [
    "hr_policies/raw/work_from_home_policy.txt",
    "hr_policies/raw/handbook.pdf",
    "hr_policies/raw/",
    "hr_policies/raw/POLICY.TXT",
]

print("endswith('.txt')")
for name in names:
    if not name.endswith(".txt"):
        print("  continue", name)
        continue
    print("  keep    ", name)

print("lower().endswith('.txt')")
for name in names:
    if not name.lower().endswith(".txt"):
        continue
    print("  keep    ", name)

---

## অংশ ৬ — ডাউনলোড

```python
text = blob.download_as_text()
```

এতক্ষণের লাইনের মধ্যে **এটাই সত্যিকারের network call**, আর এই function-এর সবচেয়ে দামি জায়গা।

`download_as_text()` বাইট নামায়, UTF-8 ধরে string বানায়। ভাই `download_as_bytes()` raw বাইট দেয় — PDF বা image-এর জন্য। `processor.py` সেটাই ব্যবহার করে।

দুইটা জিনিস মনে রাখলে পরে অবাক হতে হবে না।

**একটা file, একটা API call.** ৫০টা policy মানে ৫০ বার cloud-এ যাওয়া, একটার পর একটা। ২০টায় টের পাবে না। ৫০০০ হলে মিনিট লেগে যাবে। তখন `ThreadPoolExecutor`-এর কথা ভাবা যাবে। এখন দরকার নেই।

**সব text RAM-এ জমছে।** Loop শেষ হওয়ার আগে `documents` পুরো content ধরে রেখেছে। HR policy ছোট, তাই ঠিক আছে। Corpus যদি গিগাবাইট হতো, list-এর বদলে `yield` দিয়ে একটা করে document বের করা যেত, যাতে memory-তে একসাথে সব না থাকে।

---

## অংশ ৭ — Filename বের করা

```python
filename = blob.name.rsplit("/", 1)[-1]
```

`blob.name` পুরো path: `hr_policies/raw/work_from_home_policy.txt`। Citation-এ user-কে পুরো GCS path দেখাতে চাই না। "এই উত্তরটা `work_from_home_policy.txt` থেকে" — এইটুকুই সুন্দর।

`rsplit` ডান দিক থেকে কাটে। `1` মানে একবারই:

```python
"hr_policies/raw/wfh.txt".rsplit("/", 1)   # ['hr_policies/raw', 'wfh.txt']
```

`[-1]` শেষ টুকরো। নামে slash না থাকলেও ভয় নেই: `"wfh.txt".rsplit("/", 1)` দেয় `['wfh.txt']`, `[-1]` সেটাই ফেরত দেয়।

একটা খেয়াল: `raw/2024/leave.txt` আর `raw/2025/leave.txt` দুইটার `source` হবে একই `leave.txt`। আলাদা করতে `gcs_path` আছে, সেটা সবসময় আলাদা। নিচের cell-এ দুটোই দেখা যাবে।

In [ ]:
print("hr_policies/raw/wfh.txt".rsplit("/", 1))
print("hr_policies/raw/wfh.txt".rsplit("/", 1)[-1])
print("wfh.txt".rsplit("/", 1))
print("wfh.txt".rsplit("/", 1)[-1])

bucket_name = "myhr-policies"
for path in ("raw/2024/leave.txt", "raw/2025/leave.txt"):
    source = path.rsplit("/", 1)[-1]
    gcs_path = f"gs://{bucket_name}/{path}"
    print(f"source={source!r:12} gcs_path={gcs_path}")

---

## অংশ ৮ — Document বানানো

```python
documents.append(Document(page_content=text, metadata={
    "source": filename,
    "policy_category": extract_policy_category(text),
    "gcs_path": f"gs://{bucket_name}/{blob.name}",
}))
```

এই লাইন বাইরে থেকে পড়লে `append` আগে চোখে পড়ে। আসলে ভেতরটা আগে চলে: filename, category, gcs path তৈরি হয়, তারপর সেই Document list-এর শেষে জোড়া লাগে।

**`Document`** LangChain-এর সবচেয়ে ছোট, সবচেয়ে দরকারি জিনিস। দুইটা field:

- `page_content` — আসল text। পরে এটাই embed হবে, LLM-এর কাছে যাবে।
- `metadata` — text-এর পাশে একটা dict। উত্তর দেখানো, filter, debug — সব এখান থেকে।

| Key | কোথায় কাজে লাগবে |
|---|---|
| `source` | উত্তরে citation — "Work From Home Policy অনুযায়ী..." |
| `policy_category` | Retrieval-এ filter — WFH প্রশ্নে শুধু ওই category |
| `gcs_path` | সন্দেহ হলে আসল file-এ ফিরে যাওয়া |

`extract_policy_category(text)` সেই আগের function। পুরো text পায়, কিন্তু শুধু প্রথম ৫ line দেখে। ৩০০ line-এর policy হলেও এই খরচ প্রায় কিছুই না।

f-string-এর শুরুর `f` বলে দেয়, `{}`-এর ভেতরের জিনিস চালিয়ে ফল বসাও:

```python
f"gs://{bucket_name}/{blob.name}"
# → 'gs://myhr-policies/hr_policies/raw/work_from_home_policy.txt'
```

`gs://` GCS-এর ঠিকানা, যেমন web-এ `https://`। এই স্ট্রিং `gsutil` বা Console-এ পেস্ট করলে file খুলে যায়। Debug-এর সময় হাতের কাছে থাকা ভালো।

Metadata-র মান রাখো সরল type-এ: string, int, float, bool। Nested dict বা list দিলে Chroma বা Pinecone-এর filter বেঁকে যায় — ওরা জটিল structure বোঝে না। এখানে তিনটাই string। নিচের cell metadata আর তার type দুটোই print করে।

In [ ]:
class Document:  # LangChain Document — page_content আর metadata
    def __init__(self, page_content, metadata):
        self.page_content = page_content
        self.metadata = metadata


def extract_policy_category(text: str) -> str:
    for line in text.splitlines()[:5]:
        stripped = line.strip().lower()
        if stripped.startswith("policy category:"):
            return line.split(":", 1)[1].strip()
        if stripped.startswith("category:"):
            return line.split(":", 1)[1].strip()
    return "Unknown"


text = "WORK FROM HOME POLICY\n\nPolicy Category: Work From Home\n\n1. Purpose..."
bucket_name = "myhr-policies"
blob_name = "hr_policies/raw/work_from_home_policy.txt"
filename = blob_name.rsplit("/", 1)[-1]

documents = []
documents.append(Document(page_content=text, metadata={
    "source": filename,
    "policy_category": extract_policy_category(text),
    "gcs_path": f"gs://{bucket_name}/{blob_name}",
}))

doc = documents[0]
print(doc.metadata)
print({key: type(value).__name__ for key, value in doc.metadata.items()})
print(repr(doc.page_content[:48]))

---

## একটা file কীভাবে Document হয়

ধরো bucket `myhr-policies`, prefix `hr_policies/raw/`, আর blob `work_from_home_policy.txt`। একটা ঘূর্ণনে যা হয়:

| ধাপ | মান |
|---|---|
| `blob.name` | `'hr_policies/raw/work_from_home_policy.txt'` |
| `.endswith(".txt")` | `True`, তাই `continue` হয় না |
| `download_as_text()` | `'WORK FROM HOME POLICY\n\nPolicy Category: Work From Home\n\n1. Purpose...'` |
| `rsplit("/", 1)[-1]` | `'work_from_home_policy.txt'` |
| `extract_policy_category(text)` | `'Work From Home'` |
| f-string | `'gs://myhr-policies/hr_policies/raw/work_from_home_policy.txt'` |
| `documents` | দৈর্ঘ্য ১ বাড়ে |

তিনটা file হলে `return documents` দেবে ৩টা `Document`। এরপর `splitter.py` (06) সেগুলোকে chunk-এ ভাঙবে। নিচের cell এই টেবিলটাই print করে — আগের cell-এর `Document` আর `extract_policy_category` ব্যবহার করে, তাই ওটা আগে চালাও।

In [ ]:
bucket_name = "myhr-policies"
blob_name = "hr_policies/raw/work_from_home_policy.txt"
text = "WORK FROM HOME POLICY\n\nPolicy Category: Work From Home\n\n1. Purpose..."

kept = blob_name.endswith(".txt")
filename = blob_name.rsplit("/", 1)[-1]
category = extract_policy_category(text)
gcs_path = f"gs://{bucket_name}/{blob_name}"

documents = []
if kept:
    documents.append(Document(
        page_content=text,
        metadata={"source": filename, "policy_category": category, "gcs_path": gcs_path},
    ))

print(f"{'blob.name':<28} {blob_name!r}")
print(f"{'endswith(.txt)':<28} {kept} → continue হলো না")
print(f"{'download_as_text()':<28} {text!r}")
print(f"{'rsplit':<28} {filename!r}")
print(f"{'extract_policy_category':<28} {category!r}")
print(f"{'f-string':<28} {gcs_path!r}")
print(f"{'len(documents)':<28} {len(documents)}")

---

## যা এখন ভালো আছে, আর যা পরে যোগ করবে

যা ইতিমধ্যে পরিষ্কার:

- Default argument থাকায় test আর production একই function।
- `.txt` চেক download-এর আগে, তাই PDF নামিয়ে সময় নষ্ট হয় না।
- `gcs_path` unique, তাই একই filename-এর দুই file গুলিয়ে যায় না।
- এই function শুধু পড়ে। Parse-এর দায় `processor.py`-এর।

যা এখনও নেই, আর সেটা জেনে রাখা ভালো:

- `try/except` নেই। ১০০টা file-এর ৫৭ নম্বরে permission error হলে পুরো function থেমে যায়, আগের ৫৬টাও হারিয়ে যায়।
- Logging নেই। Prefix ভুল হলে `[]` ফিরবে, pipeline চুপচাপ এগোবে। ভুলটা ধরা পড়বে অনেক পরে, যখন retrieval কিছু খুঁজে পায় না।

তাই notebook-এ এই অভ্যাসটা রাখো। `0 documents` দেখলেই বুঝবে prefix-এ গণ্ডগোল, vector store ঘাঁটতে হবে না।

এই project-এ খালি ডাকলে prefix হয় `raw/myhr-rag-data/` — সেটা `config.GCS_PREFIX`। নিচের cell আসল cloud call। Credential লাগবে। `DefaultCredentialsError` এলে উপরে ADC-র তিন ধাপের একটাও মেলেনি। `gcloud auth application-default login` চালিয়ে cell-টা আবার চালাও।

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
for candidate in (root, *root.parents):
    if (candidate / "myhr_rag" / "document_loader.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from myhr_rag.document_loader import load_documents_from_gcs

docs = load_documents_from_gcs()
print(f"loaded {len(docs)} documents")
for d in docs:
    print(f"  {d.metadata['source']:35} | {d.metadata['policy_category']:20} | {len(d.page_content)} chars")